In [30]:
import pandas as pd
import scanpy as sc
from pathlib import Path
import scgpt as scg


/software/envs/micromamba/envs/nw-scGPTv2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
model_dir = Path("/projects/shared/intronic_bam/scGPT_supplements/save/scGPT_human")
be1_sce = sc.read_h5ad('/projects/shared/intronic_bam/datasets/anndata/be1.h5ad')
cb_sce = sc.read_h5ad('/projects/shared/intronic_bam/datasets/anndata/cord_blood.h5ad')
mix_sce = sc.read_h5ad('/projects/shared/intronic_bam/datasets/anndata/sc_mix.h5ad')

In [21]:
be1_sce

AnnData object with n_obs × n_vars = 29128 × 36753
    obs: 'Sample', 'Barcode', 'sum', 'detected', 'subsets_Mito_sum', 'subsets_Mito_detected', 'subsets_Mito_percent', 'total', 'discard', 'is_train'
    var: 'ID', 'Symbol', 'Type', 'gene_names'
    uns: 'Samples'
    layers: 'counts'

In [22]:
cb_sce

AnnData object with n_obs × n_vars = 7858 × 20400
    obs: 'adt.discard', 'mito.discard', 'discard', 'species', 'celltype', 'markers', 'sum', 'detected', 'subsets_Mito_sum', 'subsets_Mito_detected', 'subsets_Mito_percent', 'altexps_scADT_sum', 'altexps_scADT_detected', 'altexps_scADT_percent', 'total', 'is_train'
    var: 'gene_names'
    layers: 'counts'

In [23]:
mix_sce

AnnData object with n_obs × n_vars = 3918 × 11786
    obs: 'unaligned', 'aligned_unmapped', 'mapped_to_exon', 'mapped_to_intron', 'ambiguous_mapping', 'mapped_to_ERCC', 'mapped_to_MT', 'number_of_genes', 'total_count_per_cell', 'non_mt_percent', 'non_ribo_percent', 'outliers', 'is_cell_control', 'total_features_by_counts', 'log10_total_features_by_counts', 'total_counts', 'log10_total_counts', 'pct_counts_in_top_50_features', 'pct_counts_in_top_100_features', 'pct_counts_in_top_200_features', 'pct_counts_in_top_500_features', 'total_features', 'log10_total_features', 'pct_counts_top_50_features', 'pct_counts_top_100_features', 'pct_counts_top_200_features', 'pct_counts_top_500_features', 'cell_line', 'cell_line_demuxlet', 'demuxlet_cls', 'sum', 'detected', 'subsets_Mito_sum', 'subsets_Mito_detected', 'subsets_Mito_percent', 'total', 'discard', 'mito_down5pct', 'high_mito', 'is_train'
    var: 'is_feature_control', 'mean_counts', 'log10_mean_counts', 'n_cells_by_counts', 'pct_dropout_by

In [24]:
be1_ct_key = "Sample"
cb_ct_key = "celltype"
mix_ct_key = "cell_line"
gene_col = "gene_names" # name of the vector in the anndata.var with the gene names 

In [27]:
be1_sce.X = be1_sce.layers['counts']

normalized_data = sc.pp.normalize_total(be1_sce, target_sum=1e4)
sc.pp.log1p(be1_sce)

In [28]:
cb_sce.X = cb_sce.layers['counts']

normalized_data = sc.pp.normalize_total(cb_sce, target_sum=1e4)
sc.pp.log1p(cb_sce)

mix_sce already has logcounts

In [42]:
mix_sce.X = mix_sce.layers["logcounts"].copy()
mix_sce.var['gene_names'] = mix_sce.var_names

# Create embeddings for each dataset

In [34]:
be1_embed = scg.tasks.embed_data(
    be1_sce,
    model_dir,
    gene_col=gene_col,
    batch_size=64
)
be1_embed

scGPT - INFO - match 24288/36753 genes in vocabulary of size 60697.


/software/envs/micromamba/envs/nw-scGPTv2/lib/python3.11/site-packages/scgpt/model/model.py:70: UserWarning: flash-attn is not installed, using pytorch transformer instead. Set use_fast_transformer=False to avoid this warning. Installing flash-attn is highly recommended.
  warnings.warn(
/software/envs/micromamba/envs/nw-scGPTv2/lib/python3.11/site-packages/scgpt/tasks/cell_emb.py:120: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=True):
Embedding cells: 100%|██████████| 456/456 [02:02<00:00,  3.73it/s]


In [35]:
cb_embed = scg.tasks.embed_data(
    cb_sce,
    model_dir,
    gene_col=gene_col,
    batch_size=64
)
cb_embed

scGPT - INFO - match 17769/20400 genes in vocabulary of size 60697.


/software/envs/micromamba/envs/nw-scGPTv2/lib/python3.11/site-packages/scgpt/model/model.py:70: UserWarning: flash-attn is not installed, using pytorch transformer instead. Set use_fast_transformer=False to avoid this warning. Installing flash-attn is highly recommended.
  warnings.warn(
/software/envs/micromamba/envs/nw-scGPTv2/lib/python3.11/site-packages/scgpt/tasks/cell_emb.py:120: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=True):
Embedding cells: 100%|██████████| 123/123 [00:26<00:00,  4.70it/s]
/software/envs/micromamba/envs/nw-scGPTv2/lib/python3.11/site-packages/scgpt/tasks/cell_emb.py:279: ImplicitModificationWarning: Setting element `.obsm['X_scGPT']` of view, initializing view as actual.
  adata.obsm["X_scGPT"] = cell_embeddings


AnnData object with n_obs × n_vars = 7858 × 17769
    obs: 'adt.discard', 'mito.discard', 'discard', 'species', 'celltype', 'markers', 'sum', 'detected', 'subsets_Mito_sum', 'subsets_Mito_detected', 'subsets_Mito_percent', 'altexps_scADT_sum', 'altexps_scADT_detected', 'altexps_scADT_percent', 'total', 'is_train'
    var: 'gene_names', 'id_in_vocab'
    uns: 'log1p'
    obsm: 'X_scGPT'
    layers: 'counts'

In [38]:
cb_embed.obsm["X_scGPT"]

array([[-0.00159489, -0.02253452, -0.01056603, ..., -0.04902413,
        -0.00686894,  0.05470871],
       [-0.08708297, -0.00929906, -0.0001213 , ...,  0.02851844,
         0.01623704, -0.00318311],
       [ 0.00162931, -0.01645098, -0.01107094, ..., -0.0392437 ,
        -0.01197952,  0.03266908],
       ...,
       [ 0.02367991,  0.00430951, -0.02519187, ...,  0.00560382,
        -0.00400579,  0.02351748],
       [ 0.02717572, -0.0101408 , -0.00541876, ...,  0.02480123,
        -0.01617544,  0.02138694],
       [ 0.02225101,  0.00798481, -0.01182131, ...,  0.00782513,
        -0.00566659,  0.02782622]], shape=(7858, 512), dtype=float32)

In [43]:
mix_embed = scg.tasks.embed_data(
    mix_sce,
    model_dir,
    gene_col=gene_col,
    batch_size=64
)
mix_embed

scGPT - INFO - match 10900/11786 genes in vocabulary of size 60697.


/software/envs/micromamba/envs/nw-scGPTv2/lib/python3.11/site-packages/scgpt/model/model.py:70: UserWarning: flash-attn is not installed, using pytorch transformer instead. Set use_fast_transformer=False to avoid this warning. Installing flash-attn is highly recommended.
  warnings.warn(
/software/envs/micromamba/envs/nw-scGPTv2/lib/python3.11/site-packages/scgpt/tasks/cell_emb.py:120: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=True):
Embedding cells: 100%|██████████| 62/62 [00:16<00:00,  3.73it/s]
/software/envs/micromamba/envs/nw-scGPTv2/lib/python3.11/site-packages/scgpt/tasks/cell_emb.py:279: ImplicitModificationWarning: Setting element `.obsm['X_scGPT']` of view, initializing view as actual.
  adata.obsm["X_scGPT"] = cell_embeddings


AnnData object with n_obs × n_vars = 3918 × 10900
    obs: 'unaligned', 'aligned_unmapped', 'mapped_to_exon', 'mapped_to_intron', 'ambiguous_mapping', 'mapped_to_ERCC', 'mapped_to_MT', 'number_of_genes', 'total_count_per_cell', 'non_mt_percent', 'non_ribo_percent', 'outliers', 'is_cell_control', 'total_features_by_counts', 'log10_total_features_by_counts', 'total_counts', 'log10_total_counts', 'pct_counts_in_top_50_features', 'pct_counts_in_top_100_features', 'pct_counts_in_top_200_features', 'pct_counts_in_top_500_features', 'total_features', 'log10_total_features', 'pct_counts_top_50_features', 'pct_counts_top_100_features', 'pct_counts_top_200_features', 'pct_counts_top_500_features', 'cell_line', 'cell_line_demuxlet', 'demuxlet_cls', 'sum', 'detected', 'subsets_Mito_sum', 'subsets_Mito_detected', 'subsets_Mito_percent', 'total', 'discard', 'mito_down5pct', 'high_mito', 'is_train'
    var: 'is_feature_control', 'mean_counts', 'log10_mean_counts', 'n_cells_by_counts', 'pct_dropout_by

In [44]:
be1_embed.write("/projects/shared/intronic_bam/datasets/anndata/be1_scGPT_embeddings.h5ad", compression="gzip")
cb_embed.write("/projects/shared/intronic_bam/datasets/anndata/cb_scGPT_embeddings.h5ad", compression="gzip")
mix_embed.write("/projects/shared/intronic_bam/datasets/anndata/mix_scGPT_embeddings.h5ad", compression="gzip")